<a href="https://colab.research.google.com/github/Shiveshrane/Research_paper_implementations/blob/main/PaliGemmaTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PaliGemma - Followed Umar Jamil's video for the explaination

In [2]:
import torch
from torch import nn
from typing import Dict, List,Optional, Union, Tuple, Iterable
import numpy as np
from PIL import Image
import math

# Modelling SigLip

## Siglip config

In [3]:
class SiglipConfig():
  def __init__(self,
               hidden_dim=768,
               intermediate_size=3072,
               num_hidden_layers=12,
               num_attention_heads=12,
               num_channels=3,
               image_size=224,
               patch_size=16,
               layer_norm_eps=1e-6,
               attn_dropout=0.0,
               num_img_tokens:int=None,
               **kwargs
               ):
    super().__init__()
    self.hidden_dim = hidden_dim
    self.intermediate_size = intermediate_size
    self.num_hidden_layers = num_hidden_layers
    self.num_attention_heads = num_attention_heads
    self.num_channels=num_channels
    self.image_size=image_size
    self.patch_size=patch_size
    self.layer_norm_eps=layer_norm_eps
    self.attn_dropout=attn_dropout
    self.num_img_tokens=num_img_tokens

## Siglip image embedding

In [4]:
class SigLipEmbedding(nn.Module):
  def __init__(self, config=SiglipConfig()):
    super().__init__()
    self.config = config
    self.embed_size=config.hidden_dim
    self.patch_size=config.patch_size
    self.image_size=config.image_size

    self.patch_embeddings=nn.Conv2d(
        in_channels=config.num_channels,
        out_channels=self.embed_size,
        kernel_size=self.patch_size,
        stride=self.patch_size,
      padding="valid"
    )

    self.num_patches=(self.image_size//self.patch_size)**2
    self.num_positions=self.num_patches
    self.position_embedding=nn.Embedding(self.num_positions, self.embed_size)
    self.register_buffer(
        "position_ids",
        torch.arange(self.num_positions).expand((1,-1)),
        persistent=False
    )

  def forward(self, pixel_values):
    _,_, Height, Width=pixel_values.shape
    patch_embeddings=self.patch_embeddings(pixel_values)
    patch_embeddings=patch_embeddings.flatten(2).transpose(1,2)
    embeddings=patch_embeddings+self.position_embedding(self.position_ids)
    return embeddings



In [5]:
class SigLipAttention(nn.Module):
  def __init__(self, config=SiglipConfig()):
    super().__init__()
    self.config=config
    self.embed_dims=config.embed_dims
    self.num_attention_heads=config.num_attention_heads
    self.attention_head_dim=self.embed_dims//self.num_attention_heads
    self.dropout=config.attn_dropout

    self.Wq=nn.Linear(in_features=self.embed_dims, out_features=self.embed_dims)
    self.Wk=nn.Linear(in_features=self.embed_dims, out_features=self.embed_dims)
    self.Wv=nn.Linear(in_features=self.embed_dims, out_features=self.embed_dims)
    self.out=nn.Linear(in_features=self.embed_dims, out_features=self.embed_dims)

    self.dropout=nn.Dropout(p=self.dropout)
  def forward(self, hidden_states)->Tuple[torch.Tensor, Optional[torch.Tensor]]:
    batch_size, num_len, embed_dim=hidden_states.shape
    Q_states=self.Wq(hidden_states)
    K_states=self.Wk(hidden_states)
    V_states=self.Wv(hidden_states)

    Q_states=Q_states.view(batch_size, num_len, self.num_attention_heads, self.attention_head_dim)
    K_states=K_states.view(batch_size, num_len, self.num_attention_heads, self.attention_head_dim)
    V_states=V_states.view(batch_size, num_len, self.num_attention_heads, self.attention_head_dim)

    Q_states=Q_states.transpse(1,2)
    K_states=K_states.transpose(1,2) # B,l,h,s
    V_states=V_states.transpose(1,2)

    attn_score=torch.matmul(Q_states, K_states.transpose(-1,-2))/torch.sqrt(self.attention_head_dim)
    attn_score=nn.functional.softmax(attn_score, dim=-1, dtype=torch.float32).to(Q_states.dtype)
    attn_weights=self.dropout(attn_score)
    attn_score=torch.matmul(attn_weights, V_states)
    attn_score=attn_score.transpose(1,2)
    attn_score=attn_score.view(batch_size, num_len, self.embed_dims)
    attn_score=self.out(attn_score)
    return attn_score, attn_weights

In [6]:
class SigLipMLP(nn.Module):
  def __init__(self, config=SiglipConfig()):
    super().__init__()
    self.fc1=nn.Linear(config.hidden_size, config.intermediate_size)
    self.fc2=nn.Linear(config.intermediate_size, config.hidden_size)
  def forward(self, x):
    x=self.fc1(x)
    x=nn.functional.gelu(x,approximate='tanh')
    x=self.fc2(x)
    return x

In [7]:
class SigLipEncoderLayer(nn.Module):
  def __init__(self, config=SiglipConfig()):
    super().__init__()
    self.config=config
    self.attention=SigLipAttention(config)
    self.mlp=SigLipMLP(config)
    self.layer_norm1=nn.LayerNorm(config.hidden_dim, eps=config.layer_norm_eps)
    self.layer_norm2=nn.LayerNorm(config.hidden_dim, eps=config.layer_norm_eps)
  def forward(self, hidden_states):
    residual=hidden_states
    hidden_states=self.layer_norm1(hidden_states)
    attn_output,_=self.attention(hidden_states)
    hidden_states=residual+attn_output
    residual=hidden_states
    hidden_states=self.layer_norm2(hidden_states)
    mlp_output=self.mlp(hidden_states)
    hidden_states=residual+mlp_output
    return hidden_states


In [8]:
class SigLipEncoder(nn.Module):
  def __init__(self, config=SiglipConfig()):
    super().__init__()
    self.config=config
    self.layer_norm_eps=config.layer_norm_eps
    self.layers=[SigLipEncoderLayer(self.config) for _ in range(config.num_hidden_layers)]
  def forward(self, input_states):
    hidden_states=input_states
    for layer in self.layers:
      hidden_states=layer(hidden_states)
    return hidden_states

In [9]:
class SigLipVIT(nn.Module):
  def __init__(self, config=SiglipConfig()):
    super().__init__()
    self.config=config
    self.embeddings=SigLipEmbedding(config)
    self.encoder=SigLipEncoder(config)
    self.post_layer_norm=nn.LayerNorm(epsilon=config.layer_norm_eps)
  def forward(self, pixel_values):
    embeddings=self.embeddings(pixel_values)
    hidden_states=self.encoder(embeddings)
    hidden_states=self.post_layer_norm(hidden_states)
    return hidden_states

# Gemma

## Processing PaliGemma

In [10]:
IMAGENET_STANDARD_MEAN=[0.5,0.5,0.5]
IMAGENET_STANDARD_STD=[0.5,0.5,0.5]

In [11]:
def resize(image, size, resample=None, reduce_gap=None):
  h,w=size
  return image.resize((w,h), resample=resample, reducing_gap=reduce_gap)

In [12]:
def normalize(image, mean, std):
  mean=np.array(mean, dtype=image.dtype)
  std=np.array(std, dtype=image.dtype)
  image=(image-mean)/std
  return image



In [13]:
def rescale(image:np.ndarray, scale:float, dtype:np.dtype=np.float32):
  return (image*scale).astype(dtype)

In [14]:
def process_images(
    images:List[Image.Image],
    size:Dict[str, int]=None,
    resample:Image.Resampling=None,
    rescale_factor:float=None,
    image_mean:Optional[Union[float, List[float]]]=None,
    image_std:Optional[Union[float, List[float]]]=None,
):
  height=size[0]
  width=size[1]

  images=[np.array(image) for image in images]
  images=[rescale(image, scale=rescale_factor) for image in images]
  images=[normalize(image, mean=image_mean, std=image_std) for image in images]
  return images


In [15]:
def add_image_tokens_to_prompt(prefix_prompt, bos_token, image_seq_len, image_token):
  return f"{image_token*image_seq_len}{bos_token}{prefix_prompt}\n" # WE apply image_token*seq_len to reserve space for that image embedding. then we add BOS and then prompt.
  #This is then pushed to tokenizer for tokenization.

In [16]:
class PaliGemmaProcessor:
  IMAGE_TOKEN="<image>"
  def __init__(self, tokenizer, num_image_tokens, img_size):
    super().__init__()
    self.tokenizer = tokenizer
    self.num_image_tokens = num_image_tokens
    self.img_size = img_size

    tokens_to_add={"additional_special_tokens": [self.IMAGE_TOKEN]}
    tokenizer.add_special_tokens(tokens_to_add)
    EXTRA_TOKENS=[
        f"<loc{i:03d}" for i in range(1024)
    ]
    EXTRA_TOKENS+=[
        f"<seg{i:03d}" for i in range(128)
    ]
    tokenizer.add_tokens(EXTRA_TOKENS)
    self.image_token_id=tokenizer.convert_tokens_to_ids(self.IMAGE_TOKEN)
    tokenizer.add_bos_token=False
    tokenizer.add_eos_token=False
    self.tokenizer=tokenizer

  def __call__(self, text:List[str], images:List[Image.Image], padding:str="longest", truncation:bool=True):
    assert len(images)==1 and len(text)==1, f"recieved {len(images)} images fr {len(text)} prompts"
    pixel_values=process_images(
        images,
        size=(self.img_size, self.img_size),
        resample=Image.Resampling.BICUBIC,
        rescale_factor=1/255.,
        image_mean=IMAGENET_STANDARD_MEAN,
        image_std=IMAGENET_STANDARD_STD
    )
    pixel_values=np.stack(pixel_values)
    pixel_values=torch.from_numpy(pixel_values)

    input_strings=[
        add_image_tokens_to_prompt(
            prompt,
            self.tokenizer.bos_token,
            self.num_image_tokens,
            self.IMAGE_TOKEN
        ) for prompt in text
    ]
    inputs=self.tokenizer(
        input_strings,
        padding=padding,
        truncation=truncation,
        return_tensors="pt"
        )
    return_data={
        "pixel_values":pixel_values,
        **inputs
    }
    return return_data

## Modelling Gemma

### Configs

In [17]:
class GemmaConfig():
  def __init__(
      self,
      vocab_size,
      hidden_size,
      intermediate_size,
      num_hidden_layers,
      num_attention_heads,
      num_key_value_heads,
      head_dim=256,
      max_position_embeddings=8192,
      rms_norm_eps=1e-6,
      rope_theta=10000.0,
      attention_bias=False,
      attention_dropout=0.0,
      pad_token_id=None,
      **kwargs
  ):
    super().__init__()
    self.vocab_size=vocab_size
    self.max_position_embeddings=max_position_embeddings
    self.hidden_size=hidden_size
    self.intermediate_size=intermediate_size
    self.num_hidden_layers=num_hidden_layers
    self.num_attention_heads=num_attention_heads
    self.num_key_value_heads=num_key_value_heads
    self.head_dim=head_dim
    self.rms_norm_eps=rms_norm_eps
    self.rope_theta=rope_theta
    self.attention_bias=attention_bias
    self.attention_dropout=attention_dropout
    self.pad_token_id=pad_token_id

In [18]:
class PaliGemmaConfig():

   def __init__( self,
    vision_config=None,
    text_config=None,
    ignore_index=-100,
    image_token_index=256000,
    vocab_size=257152,
    projection_dim=2048,
    hidden_size=2048,
    pad_token_id=None,
    **kwargs):
    super().__init__()
    self.ignore_index=ignore_index
    self.vocab_size=vocab_size
    self.projection_dim=projection_dim
    self.hidden_size=hidden_size
    self.pad_token_id=pad_token_id
    self.vision_config=vision_config
    self.text_config=text_config
    self.image_token_index=image_token_index
    self.vision_config=vision_config
    self.is_encoder_decoder=False
    self.vision_config=SiglipConfig(**vision_config)
    self.text_config=GemmaConfig(**text_config, pad_token_id=pad_token_id)
    self.vocab_size=self.text_config.vocab_size

    self.text_config.num_image_tokens=(self.vision_config.image_size//self.vision_config.patch_size)**2
    self.vision_config.projection_dim=projection_dim



### Rotary Embeddings


This positional embeddings is made for a particular use case, where dynamic and memory efficiency is top priority. The whole method is similar to the original RoPE, but we use this for long sequences, where we only find out positional_embeddings of current **Position_ids**.

In [19]:
class GemmaRotaryEmbedding(nn.Module):
  def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
    super().__init__()
    self.dim=dim
    self.max_position_embeddings=max_position_embeddings
    self.base=base
    self.device=device
    inv_freq=1.0/(self.base**(torch.arange(0, self.dim, 2).float().to(device)/self.dim))

    self.register_buffer("inv_freq", inv_freq, persistent=False)

  @torch.no_grad()
  def forward(self, x, position_ids, seq_len=None, **kwargs):
    self.inv_freq.to(x.device)
    inv_freq_expanded=self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
    position_ids_expanded=position_ids[:, None, :].float()
    device_type=x.device
    device_type=device_type if isinstance(device_type, str) and device_type!="mps" else "cpu"
    with torch.autocast(device_type, enabled=False):
      freqs=(inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1,2)
      emb=torch.cat((freqs, freqs), dim=-1)
      cos=torch.cos(emb)
      sin=torch.sin(emb)
    return cos.to(x.dtype), sin.to(x.dtype)

In [20]:
def rotate_half(x):
  x1=x[..., : x.shape[-1]//2]
  x2=x[..., x.shape[-1]//2:]
  return torch.cat((-x2, x1), dim=-1)

In [21]:
def apply_rotary_pos_emb(q,k,cos,sin,unsqueeze_dim=1):
  cos=cos.to(q.dtype)
  sin=sin.to(q.dtype)
  cos=cos.unsqueeze(unsqueeze_dim)
  sin=sin.unsqueeze(unsqueeze_dim)
  q_embed=(q*cos)+(rotate_half(q)*sin)
  k_embed=(k*cos)+(rotate_half(k)*sin)
  return q_embed, k_embed

### KV_Cache

In [22]:
class KVCache():
  def __init__(self):
    self.key_cache=[]
    self.value_cache=[]

  def num_items(self):
    if len(self.key_cache)==0:
      return 0
    else:
      return self.key_cache[0].shape[-2]

  def update(self, key,value,layer_idx):
    if len(self.key_cache)<=layer_idx: #the kv cache of that layer is empty, so we create it by directly appending
      self.key_cache.append(key)
      self.value_cache.append(value)
    else:
      self.key_cache[layer_idx]=torch.cat([self.key_cache[layer_idx], key], dim=-2) ## We do the updation along the sequence dim, since the sentence grows along sequence, hence DIM=-2
      self.value_cache[layer_idx]=torch.cat([self.value_cache[layer_idx], value], dim=-2)
    return self.key_cache[layer_idx], self.value_cache[layer_idx]

### RMS

In [23]:
class RMSNorm(nn.Module):
  def __init__(self, hidden_size, eps=1e-6):
    super().__init__()
    self.eps=eps
    self.hidden_size=hidden_size
    self.weight=nn.Parameter(torch.zeros(self.hidden_size))
  def _norm(self, hidden_states):
    return hidden_states*torch.rsqrt(torch.mean(hidden_states**2, dim=-1, keepdim=True)+self.eps)
  def forward(self, hidden_states):
    output=self._norm(hidden_states.float())
    output= output*(1.0+self.weight.float())
    return output.astype(hidden_states)

### Gemma MLP

In [24]:
class GemmaMLP(nn.Module):
  def __init__(self, config:GemmaConfig):
    self.config=config
    self.hidden_size=config.hidden_size
    self.intermediate_size=config.intermediate_size
    self.up_proj=nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
    self.gated_proj=nn.Linear(self.hidden_size, self.intermediate_size, bias=False)
    self.down_proj=nn.Linear(self.intermediate_size, self.hidden_size, bias=False)

  def forward(self, hidden_states):
    up_proj_state=self.up_proj(hidden_states)
    gated_proj_state=nn.functional.gelu(self.gated_proj(hidden_states), approximate='tanh')
    down_proj_state=self.down_proj(gated_proj_state*up_proj_state)
    return down_proj_state

### Gemma Attention

In [25]:
def kv_repeat(state, number):
  batch_size, num, s_len, embed_dim=state.shape
  if number==1:
    return state
  state=state[:,:,None,:,:].expand(batch_size, num, number, s_len, embed_dim)
  return state.reshape(batch_size, num*number, s_len, embed_dim)

In [26]:
class GemmaAttention(nn.Module):
  def __init__(self, layer_idx,config=GemmaConfig):
    self.config=GemmaConfig
    self.hidden_size=config.hidden_size
    self.num_attention_heads=config.num_attention_heads
    self.num_key_value_heads=config.num_key_value_heads
    self.head_dim=config.head_dim
    self.key_value_groups=self.num_attention_heads//self.num_key_value_heads
    self.max_position_embeddings=config.max_position_embeddings
    self.attention_dropout=config.attention_dropout
    self.attention_bias=config.attention_bias
    self.layer_idx=layer_idx

    self.rope_theta=config.rope_theta
    self.is_causal=True
    assert self.hidden_size%self.num_attention_heads==0, "hidden_size must be divisible by num_attention_heads"

    self.Wq=nn.Linear(self.hidden_size, self.hidden_size, bias=False)
    self.Wk=nn.Linear(self.hidden_size, self.hidden_size, bias=False)
    self.Wv=nn.Linear(self.hidden_size, self.hidden_size, bias=False)
    self.out=nn.Linear(self.hidden_size, self.hidden_size, bias=False)

    self.rotary_emb=GemmaRotaryEmbedding(
        self.head_dim,
        max_position_embeddings=self.max_position_embeddings,
        base=self.rope_theta
    )

  def forward(self, hidden_states, attention_mask, position_ids, kv_cache):
    batch,q_len,_=hidden_states.size()
    q_state=self.Wq(hidden_states)
    k_state=self.Wk(hidden_states)
    v_state=self.Wv(hidden_states)

    q_states=q_states.view(batch, q_len, self.num_attention_heads, self.head_dim).transpose(1,2)
    k_states=k_states.view(batch, q_len, self.num_key_value_heads, self.head_dim).transpose(1,2)
    v_states=v_states.view(batch, q_len, self.num_key_value_heads, self.head_dim).transpose(1,2)

    cos,sin=self.rotary_emb(v_state, position_ids, seq_len=None)
    q_states, k_state=apply_rotary_pos_emb(q_states, k_states, cos, sin)
    if kv_cache is not None:
      k_state, v_state=kv_cache.update(k_state, v_state, self.layer_idx)
    k_state=kv_repeat(k_state, self.key_value_groups)
    v_state=kv_repeat(v_state, self.key_value_groups)

    attn_score=torch.matmul(q_states, k_state.transpose(2,3))/torch.sqrt(self.head_dim)
    assert attention_mask is not None
    attn_score=attn_score+attention_mask
    attn_score=nn.functional.softmax(attn_score, dim=-1, dtype=torch.float32).to(q_states.dtype)
    attn_score=nn.functional.dropout(attn_score, p=self.attention_dropout, training=self.training)
    attn_output=torch.matmul(attn_score, v_state)
    attn_output=attn_output.transpose(1,2).contiguous()
    attn_output=attn_output.view(batch,q_len, self.hidden_size)
    attn_output=self.out(attn_output)
    return attn_output, attn_score


### DecoderLayer

In [27]:
class GemmaDecoderLayer(nn.Module):
  def __init__(self, config:GemmaConfig, layer_idx):
    super().__init__()
    self.config=config
    self.layer_idx=layer_idx
    self.hidden_size=config.hidden_size
    self.mlp=GemmaMLP(config)
    self.input_layernorm=RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
    self.postattn_layernorm=RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
    self.self_attn=GemmaAttention(config, layer_idx)
  def forward(self, hidden_states, attention_mask=None, position_ids=None, kv_cache=None):
    residual=hidden_states
    hidden_states=self.input_layernorm(hidden_states)
    attn_output, attn_weights=self.self_attn(hidden_states, attention_mask, position_ids, kv_cache)
    hidden_states=residual+attn_output
    residual=hidden_states
    hidden_states=self.postattn_layernorm(hidden_states)
    mlp_output=self.mlp(hidden_states)
    hidden_states=residual+mlp_output
    return hidden_states

### Gemma Model

In [28]:
class GemmaModel(nn.Module):
  def __init__(self,config:GemmaConfig):
    super().__init__()
    self.config=config
    self.padding_idx=config.pad_token_id
    self.vocab_size=config.vocab_size
    self.embed_tokens=nn.Embedding(self.vocab_size, config.hidden_size, padding_idx=self.padding_idx)
    self.layers=nn.ModuleList(
        [
            GemmaDecoderLayer(config, layer_idx) for layer_idx in range(config.num_hidden_layers)
        ]
    )
    self.norm=RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

  def get_input_embeddings(self):
    return self.embed_tokens
  def forward(
      self,
      attention_mask:Optional[torch.Tensor]=None,
      position_ids:Optional[torch.Tensor]=None,
      input_embeds:Optional[torch.Tensor]=None,
      kv_cache:Optional[KVCache]=None
  ):
    hidden_states=input_embeds
    normalizer=torch.tensor(self.config.hidden_size**0.5, dtype=hidden_states.dtype)
    hidden_states=hidden_states*normalizer
    for layer in self.layers:
      hidden_states=layer(
          hidden_states,
          attention_mask=attention_mask,
          position_ids=position_ids,
          kv_cache=kv_cache
      )
      hidden_states=self.norm(hidden_states)
    return hidden_states

### PaliGemma MultimodalProjector

In [29]:
class PaliGemmaMultiModalProjector(nn.Module):
  def __init__(self, config:PaliGemmaConfig):
    super().__init__()
    self.config=config
    self.projector=nn.Linear(config.vision_config.hidden_dim, config.vision_config.projection_dim, bias=True)
  def forward(self, image_features):
    return self.projector(image_features)

### Gemma for Causal LM

In [30]:
class GemmaForCausalLM(nn.Module):
  def __init__(self, config:GemmaConfig):
    super().__init__()
    self.config=config
    self.model=GemmaModel(config)
    self.vocab_size=config.vocab_size
    self.lm_head=nn.Linear(config.hidden_size, config.vocab_size, bias=False)

  def get_input_embeddings(self):
    return self.model.embed_tokens
  def tie_weights(self):
    self.lm_head.weight=self.model.embed_tokens.weight

  def forward(self,
              attention_mask:Optional[torch.Tensor]=None,
              position_ids:Optional[torch.Tensor]=None,
              input_embeds:Optional[torch.Tensor]=None,
              kv_cache:Optional[KVCache]=None):
              outputs=self.model(
                  attention_mask=attention_mask,
                  position_ids=position_ids,
                  input_embeds=input_embeds,
                  kv_cache=kv_cache
              )
              hidden_states=outputs
              lm_logits=self.lm_head(hidden_states)
              lm_logits=lm_logits.float()
              return_data={
                  "logits":lm_logits
              }
              if kv_cache is not None:
                return_data["kv_cache"]=kv_cache

              return return_data

## Complete model

In [31]:
class PaliGemmaForConditionalGeneration(nn.Module): # Connector class (Connects complete model)
  def __init__(self, config:PaliGemmaConfig):
    super().__init__()
    self.config=config
    self.vision_tower=SigLipVIT(config.vision_config)
    self.multi_modal_projector=PaliGemmaMultiModalProjector(config)
    self.vocab_size=config.vocab_size
    language_model=GemmaForCausalLM(config.text_config)
    self.language_model=language_model
    self.pad_token=self.config.pad_token_id if self.config.pad_token_id is not None else -1

  def tie_weights(self): #
    return self.language_model.tie_weights()


  def merge_ids_with_image_features(self, image_features: torch.Tensor, input_embeds:torch.Tensor, input_ids: torch.Tensor, attention_mask:torch.Tensor, kv_cache:Optional[KVCache]=None):
    batch_size, num_patches, embed_dim=image_features.shape
    text_batch_size, seq_len=input_ids.shape
    dtype, device=input_embeds.dtype, input_embeds.device
    #Scale the image feature
    scaled_image_features=image_features/(self.config.hidden_size**0.5)

    #final output tensor for output
    final_embedding=torch.zeros(batch_size, seq_len, embed_dim, dtype=input_embeds.dtype, device=input_embeds.device)
    #Mask
    text_mask=(input_ids!=self.config.image_token_index)&(input_ids!=self.config.pad_token_id) ## Mask for text (not image and not pad)
    image_mask=input_ids==self.config.image_token_index #image mask
    pad_mask=input_ids==self.config.pad_token_id #pad mask

    expanded_text_mask=text_mask.unsqueeze(-1).expand(-1,-1,embed_dim)#Create a new dimension at the end, and make it equal to embed_dim using expand for torch.where()
    expanded_image_mask=image_mask.unsqueeze(-1).expand(-1,-1,embed_dim) #Create a new dimension at the end, and make it equal to embed_dim using expand for torch.where()
    expanded_pad_mask=pad_mask.unsqueeze(-1).expand(-1,-1,embed_dim) #Create a new dimension at the end, and make it equal to embed_dim using expand for torch.where()

    # So now the size goes from (B,s)--unsqueeze-->(B,S,1)--Expand-->(B,S,D)

    #Add text embedding to final embedding
    final_embedding=torch.where(expanded_text_mask, input_embeds, final_embedding)
    # Add image tokens (we cant use tch.where since the scaled image features dont have same seq len to final embedding. We use masked_scatter() )
    final_embedding=final_embedding.masked_scatter(expanded_image_mask, scaled_image_features)
    #add padding tokens (can use torch.where())
    final_embedding=torch.where(expanded_pad_mask,torch.zeros_like(final_embedding), final_embedding)





     ### attention mask (Need KKVCache for it) ### THIS IS ONLY FOR INFERENCE PART, SINCE WE USE PRETRAINED MODEL
     # It works by creating an all 0s mask, which is of the shape of q_len+kv_len, for each autoregressive iteration.
    dtype.device=input_embeds.dtype, input_embeds.device
    min_dtype=torch.finfo(dtype).min
    q_len=input_embeds.shape[1]

    if kv_cache is None or kv_cache.num_items()==0:
      causal_mask=torch.full(
          (batch_size, q_len,q_len),
          fill_value=0,
          dtype=dtype,
          device=device
      )
    else:
      assert q_len==1
      kv_len=kv_cache.num_items()+q_len
      causal_mask=torch.full(
          (batch_size, q_len, kv_len),
          fill_value=0,
          dtype=dtype,
          device=device
      )
    causal_mask=causal_mask.unsqueeze(1)


    ###this allows you to get the position id of the last token in the attention_mask
    if kv_cache is not None and kv_cache.num_items()>0:
      position_ids=attention_mask.cumsum(-1)[:,-1]
      if position_ids.dim()==1:
        position_ids=position_ids.unsqueeze(0)
    else:
      position_ids=(attention_mask.cumsum(-1)).masked_fill_((attention_mask==0),1).to(device)


    return final_embedding, causal_mask, position_ids




  def forward(self,
              input_ids: torch.LongTensor=None,
              pixel_values:torch.FloatTensor=None,
              attention_mask:Optional[torch.Tensor]=None,
              kv_cache:Optional[KVCache]=None)->Tuple:
    assert torch.all(attention_mask==1), "input cannot be padded"

    #Extract the input embeddings
    input_embeds=self.language_model.get_input_embeddings()(input_ids) # batch, seq_len, hidden_size
    # Merge image and texts
    selected_image_features=self.vision_tower(pixel_values.to(input_embeds.dtype)) # B,C,H,W-> B,Num_patches, Embed_dims
    # Convert the image embed dims to text hidden_size
    image_features=self.multi_modal_projector(selected_image_features)
    # Merge the embeddings of text and images
    input_embeds, attn_mask, position_ids=self.merge_ids_with_image_features(image_features, input_embeds, input_ids, attention_mask, kv_cache)

    #Calling the language model

    llm_op=self.language_model(
        attention_mask=attn_mask,
        position_ids=position_ids,
        input_embeds=input_embeds,
        kv_cache=kv_cache
    )
    return llm_op

## Inference

In [45]:
!pip install fire

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fire: filename=fire-0.7.0-py3-none-any.whl size=114249 sha256=62f85be5dcae906b94d246fe9dcdc758bc4e2b10c72ec7e24b20a5954a926733
  Stored in directory: /root/.cache/pip/wheels/46/54/24/1624fd5b8674eb1188623f7e8e17cdf7c0f6c24b609dfb8a89
Successfully built fire


In [47]:
from transformers import AutoTokenizer
import json
import glob
from safetensors import safe_open
from typing import Tuple
import os

def load_hf_model(model_path: str, device: str) -> Tuple[PaliGemmaForConditionalGeneration, AutoTokenizer]:
    # Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, padding_side="right")
    assert tokenizer.padding_side == "right"

    # Find all the *.safetensors files
    safetensors_files = glob.glob(os.path.join(model_path, "*.safetensors"))

    # ... and load them one by one in the tensors dictionary
    tensors = {}
    for safetensors_file in safetensors_files:
        with safe_open(safetensors_file, framework="pt", device="cpu") as f:
            for key in f.keys():
                tensors[key] = f.get_tensor(key)

    # Load the model's config
    with open(os.path.join(model_path, "config.json"), "r") as f:
        model_config_file = json.load(f)
        config = PaliGemmaConfig(**model_config_file)

    # Create the model using the configuration
    model = PaliGemmaForConditionalGeneration(config).to(device)

    # Load the state dict of the model
    model.load_state_dict(tensors, strict=False)

    # Tie weights
    model.tie_weights()

    return (model, tokenizer)

In [48]:
import fire

In [55]:
def move_inputs_to_device(model_inputs: dict, device: str):
    model_inputs = {k: v.to(device) for k, v in model_inputs.items()}
    return model_inputs


def get_model_inputs(
    processor: PaliGemmaProcessor, prompt: str, image_file_path: str, device: str
):
    image = Image.open(image_file_path)
    images = [image]
    prompts = [prompt]
    model_inputs = processor(text=prompts, images=images)
    model_inputs = move_inputs_to_device(model_inputs, device)
    return model_inputs


In [62]:
def sample_top_p(logits, p):
  prob_sort, prob_idx=torch.sort(logits, -1, descending=True)
  probs_sum=torch.cumsum(prob_sort, dim=-1)
  mask=probs_sum-prob_sort>p
  prob_sort[mask]=0
  prob_sort.div_(prob_sort.sum(dim=-1, keepdim=True))
  next_token=torch.multinomial(prob_sort, num_samples=1) # Sample the token
  next_token=torch.gather(prob_idx, -1, next_token) #Chosing the next token
  return next_token


In [61]:
def test_inference(model, processor, device, prompt, image_file_path, max_tokens_to_generate, temperature, top_p, do_sample):
  model_inputs=get_model_inputs(processor, prompt, image_file_path, device)
  input_ids=model_inputs["input_ids"]
  attention_mask=model_inputs["attention_mask"]
  pixel_values=model_inputs["pixel_values"]

  kv_cache=KVCache()

  stop_token=processor.tokenizer.eos_token_id
  generated_token=[]
  for _ in range(max_tokens_to_generate):
    outputs=model(
        input_ids,
        attention_mask=attention_mask,
        pixel_values=pixel_values,
        kv_cache=kv_cache
    )
    logits=outputs["logits"]
    kv_cache=outputs["kv_cache"]
    next_token_logits=logits[:,-1,:]
    if do_sample:
      next_token_logits=torch.softmax(next_token_logits/temperature, dim=-1) # Temperature increases diversity
      next_token=sample_top_p(next_token_logits, top_p)
    else:
      next_token=torch.argmax(next_token_logits, dim=-1)
    assert next_token.size()==(1,1)
    next_token=next_token.squeeze(0)
    generated_token.append(next_token)
    if next_token.item()==stop_token:
      break
    input_ids=next_token.unsqueeze(-1)
    attention_mask=torch.cat(
        [attention_mask, torch.ones((1,1), dtype=torch.long, device=device)], dim=-1
    )
  generated_tokens=torch.cat(generated_token, dim=-1)
  generated_text=processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)
  print(prompt + generated_text)


In [59]:
def main(
    model_path=None,
    prompt=None,
    image_file_path=None,
    max_tokens_to_generate=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=False,
    only_cpu=False
):
  device="cpu"
  if not only_cpu:
    device="cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
  model, tokenizer=load_hf_model(model_path, device)
  model.to(device).eval()
  num_image_tokens=model.config.vision_config.num_image_tokens
  image_size=model.config.vision_config.image_size
  processor=PaliGemmaProcessor(tokenizer, num_image_tokens, image_size)
  print("Running Inference:")
  with torch.no_grad():
    test_inference(
         model,
          processor,
          device,
          prompt,
          image_file_path,
          max_tokens_to_generate,
          temperature,
          top_p,
          do_sample,
    )


In [64]:
query="Enter query"
image_path="image path"
model_path="model path"
max_tokens_to_generate=100
temperature=0.8
top_p=0.9
do_sample=False
only_cpu=False

In [65]:
if __name__=="main":
  main(
      model_path=model_path,
      prompt=query,
      image_file_path=image_path,
      max_tokens_to_generate=max_tokens_to_generate,
      temperature=temperature,
      top_p=top_p,
      do_sample=do_sample,
      only_cpu=only_cpu
  )